Preprocessing by Nguyen

In [1]:
import ast
import pandas as pd
import numpy as np
import spacy
from typing import Literal
import pickle
import sys

Role = Literal[-1, 1]
# array shape: (L, V) where L is number of tokens, V is vector size (currently 50)
# None means a long pause
Text = tuple[Role, np.ndarray | None]
Dialog = list[Text]
Label = Literal[-1, 0, 1]

def get_embeddings(file: str) -> dict:
    embeddings = {}
    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split(" ")
            word = values[0]
            arr = [float(item) if item != "." else 0.0 for item in values[1:]]
            vector = np.array(arr)
            embeddings[word] = vector
    return embeddings

def preprocess_data(embeddings_file: str, out: str, file: str = "Data/train.csv") -> list[tuple[Dialog, Label]]:
    embeddings = get_embeddings(embeddings_file)
    print("Embeddings loaded with size:", len(embeddings))
    nlp = spacy.load("en_core_web_sm")
    df = pd.read_csv(file)
    data: list[tuple[Dialog, Label]] = []
    for row in df.iloc:
        dialogue = ast.literal_eval(row['Dialogue'])['text']
        label = row['Label']
        dialogue_data: Dialog = []
        for text in dialogue:
            role = 1 if text['role'] == 'A' else -1
            if text['response'] == '$S$' or text['response'].strip() == '':
                dialogue_data.append((role, None))
                continue
            if text['response'] == '$EXIT$':
                break
            doc = nlp(text['response'])
            embeddings_data: list[np.ndarray] = []
            for token in doc:
                if token.lemma_ in embeddings:
                    embeddings_data.append(embeddings[token.lemma_])
            # This may cause issues because we are ignoring sentences only with emojis
            if len(embeddings_data) == 0:
                continue
            embeddings_data = np.array(embeddings_data)
            dialogue_data.append((role, embeddings_data))
        if len(dialogue_data) > 0:
            data.append((dialogue_data, label))
    with open(out, "wb") as f:
        pickle.dump(data, f)
    return data

def load_data(file: str) -> tuple[list[tuple[Dialog, Label]], int]:
    """
    Paremeters
    ---
    file: str
        The file to load the data from.
    
    Returns
    ---
    tuple[list[tuple[Dialog, Label]], int]
        The data and the size of each word vector.
        Size of each word vector is inferred from the file name.
    """
    with open(file, "rb") as f:
        data = pickle.load(f)
    size = int(file.split(".")[-2][:-1])
    return data, size

Preprocessing by Yibai

Notation:

- sequence == sentence
- batch == dialog
- inputs == X
- labels == y
- predictions == pred_y

In [2]:
from torch.nn.utils.rnn import PackedSequence

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch import Tensor
from torch.utils.data import Dataset
from torch.nn.utils.clip_grad import clip_grad_norm_
from torch.nn.utils.rnn import pad_sequence, pad_packed_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [3]:
def to_inputs_and_labels(data: list[tuple[Dialog, Label]]) -> tuple[list[list[np.ndarray]], list[Label]]:
    '''Split data into inputs and labels

    Args:
        data(list[Dialog]): loaded training data, with non-uniform batch size
        and sequence length.  
                
    :Returns: tuple[inputs, labels] WHERE

        inputs(list[list[np.ndarray]]): input data, with non-uniform batch size
        and sequence length, ignoring Role.
        
        labels(list[Label]): labels for each dialog
    '''
    inputs = []
    labels = []
    for batch in data:
        inputs_batch = []
        labels.append(batch[1])
        for pair in batch[0]:
            inputs_batch.append(pair[1])
        inputs.append(inputs_batch)
    return (inputs, labels)

In [ ]:
def is_valid_batch(
    inputs: list[np.ndarray],
    labels: list[Literal[-1, 1]],
    embed_size: int
) -> bool:
    '''Verify whether the inputs and labels of a batch have valid vector representations.

    Args:
        inputs(list[np.ndarray]): list of all sequences in the batch
        labels(list[Literal[-1, 1]]): list of labels for each sequence in the batch
        embed_size(int): token embedding dimension
    '''
    # Verify that inputs and labels are non-empty:
    if len(inputs) == 0 or len(labels) == 0:
        return False
    
    # Verify that all sequences in the batch are non-empty:
    for seq in inputs:
        if seq.shape == (0, ):
            return False
    
    # Verify that all tokens in the batch have the expected size of embedding:
    for seq in inputs:
        if seq.shape[1] != embed_size:
            return False
    
    # Verify that inputs and labels have compatible dimensions:
    if len(inputs) != len(labels):
        return False

    return True

def filter_good_batches(
    inputs: list[list[np.ndarray]],
    labels: list[list[Literal[-1, 1]]],
    embed_size: int
) -> tuple[list[list[np.ndarray]], list[list[Literal[-1, 1]]]]:
    '''Remove all batches containing invalid data

    Args:
        inputs(list[list[np.ndarray]]): inputs for all batches
        labels(list[list[Literal[-1, 1]]]): labels every sequences
    '''
    inputs_copy = inputs.copy()
    labels_copy = labels.copy()
    num_batch = len(inputs)
    assert(num_batch == len(labels))
    
    pop_count = 0
    for i in range(num_batch):
        if not is_valid_batch(inputs[i], labels[i], embed_size):
            inputs_copy.pop(i - pop_count)
            labels_copy.pop(i - pop_count)
            pop_count += 1
    
    return (inputs_copy, labels_copy)

In [4]:
def collate_inputs(inputs: list[list[np.ndarray]]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    r'''Pad sequences to global max_seq_len. Then pad batches to max_batch_size.

    Use this function instead of calling nn.utils.rnn.pad_sequence on each batch,
    because each batch has its own max_seq_len, and we need to use the global max.

    Args:
        inputs(list[list[np.ndarray]]): All input data
    
    Returns:
        tuple(tuple[torch.Tensor, torch.Tensor, torch.Tensor]):
        Input tensor in the shape (num_batch, max_batch_size, max_seq_len),
        seq_lens tensor in the shape (num_batch, max_batch_size), and
        batch_sizes tensor in the shape (num_batch)
    '''
    # Flatten seqs across all dialogs and convert type from np.ndarray to torch.Tensor
    all_sequences = [torch.tensor(seq) for dialog in inputs for seq in dialog]

    max_seq_len = max(seq.size(0) for seq in all_sequences)
    padded_sequences = [F.pad(seq, (0, 0, 0, max_seq_len - seq.size(0))) for seq in all_sequences]

    # Group padded sequences back into dialog structure
    num_batch = len(inputs)
    max_batch_size = max(len(dialog) for dialog in inputs)

    padded_dialogs = []
    seq_lens = []
    batch_sizes = []

    idx = 0
    for dialog in inputs:
        padded_dialogs.append(torch.stack(padded_sequences[idx:idx + len(dialog)]))
        seq_lens.append(torch.tensor([len(seq) for seq in dialog]))
        batch_sizes.append(len(dialog))
        idx += len(dialog)

    # Pad dialogs and labels with zero sequences to match max_batch_size
    for i in range(num_batch):
        while len(padded_dialogs[i]) < max_batch_size:
            padded_dialogs[i] = torch.cat((padded_dialogs[i], torch.zeros(1, max_seq_len, padded_dialogs[i].size(-1))), dim=0)
            seq_lens[i] = torch.cat((seq_lens[i], torch.tensor([0])))

    return torch.stack(padded_dialogs), torch.stack(seq_lens), torch.tensor(batch_sizes)

In [5]:
data = load_data('../Data/features/glove.6B.50d.pkl')[0]
input_dim = 50

# Data: list[tuple[Dialog, Label]]

X, y = to_inputs_and_labels(data)
# X, y = filter_good_batches(X, y, input_dim)
X, seq_lens, batch_sizes = collate_inputs(X)
# y = pad_sequence([torch.tensor(batch) for batch in y]).T
y = torch.tensor(y)
# y_mask = (y != 0)
y = y + 1
y = y.long()
X = X.float()

In [6]:
train_X, test_X, train_y, test_y, train_seq_lens, test_seq_lens, train_batch_sizes, test_batch_sizes = train_test_split(
    X, y, seq_lens, batch_sizes,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [7]:
class LstmDataset(Dataset[tuple[Tensor, Tensor, Tensor, Tensor]]):
    def __init__(self, X, y, seq_lens, batch_sizes):
        self.X = X
        self.y = y
        self.seq_lens = seq_lens
        self.batch_sizes = batch_sizes
       
    def __getitem__(self, index) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        return (X, y, seq_lens, batch_sizes)
    
    def __len__(self):
        return len(self.y)

train_data = LstmDataset(train_X, train_y, train_seq_lens, train_batch_sizes)
test_data = LstmDataset(test_X, test_y, test_seq_lens, test_batch_sizes)

Simple LSTM Model

In [9]:
class TokenLSTM(nn.Module):
    '''Token LSTM Model

        This is a model which takes in token embeddings and generate a
        seq-aware token embedding for each sequence.

        X: input, shape: (num_batch, max_batch_size, max_seq_len, token_embed_size)
        
        pred_y: output, shape: (num_batch * max_batch_size, max_seq_len, hidden_size)
    '''
    def __init__(self, input_dim, hidden_dim, num_layers):
        '''
        Args:
            input_dim: (input size) token_embed_size
            hidden_dim: (output size) hidden_size
        '''
        super(TokenLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True).float()
    
    def _pack_wrapper(self, X: torch.Tensor, seq_lens: list) -> PackedSequence:
        '''Mask the padding in input tensor to improve computational efficiency.
        This is a wrapper to specify reshaping operations involved.

        Returns:
            packed_X(torch.nn.utils.rnn.PackedSequence): packed input tensor
        '''
        flattened_X = X.view(-1, X.size(-2), X.size(-1))
        flattened_seq_lens = np.array(seq_lens).reshape(-1)

        # Filter out zero-length sequences, since pack_padded_sequence() only
        # accepts non-empty sequences.
        non_empty_mask = flattened_seq_lens > 0
        filtered_X = flattened_X[non_empty_mask]
        filtered_seq_lens = flattened_seq_lens[non_empty_mask]
        
        # Pack the sequences
        return pack_padded_sequence(filtered_X, 
                                    filtered_seq_lens, 
                                    batch_first=True,
                                    enforce_sorted=False)

    def _reintroduce_empty_seqs(self, pred_y, X, seq_lens):
        augmented_y = torch.zeros((X.size(0) * X.size(1), X.size(2), self.hidden_dim))
        non_empty_mask = np.array(seq_lens).reshape(-1) > 0
        augmented_y[non_empty_mask] = pred_y
        return augmented_y

    def forward(self, X: torch.Tensor, seq_lens: torch.Tensor):
        packed_X = self._pack_wrapper(X, seq_lens)
        packed_y, _ = self.lstm(packed_X)
        pred_y, _ = pad_packed_sequence(packed_y, batch_first=True)
        return self._reintroduce_empty_seqs(pred_y, X, seq_lens)

In [10]:
class SequenceLSTM(nn.Module):
    '''Sequence LSTM Model

        This is a model which takes in seq-aware token embeddings obtained from
        TokenLSTM and generate a dialog-aware seq embedding for each seq.

        X: input, shape: (num_batch * max_batch_size, max_seq_len, token_lstm_hidden_size)
        
        pred_y: prediction for each seq, shape: (num_batch, max_batch_size, seq_embed_size)

        dialog_embed: output, embedding of the last sequence, shape: (num_batch, seq_embed_size)
    '''
    def __init__(self, input_dim, hidden_dim, num_layers):
        '''
        Args:
            input_dim: (input size) token_lstm_hidden_size
            hidden_dim: (output size) seq_embed_size
        '''
        super(SequenceLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True).float()

    def forward(self, X, batch_sizes):
        packed_X = pack_padded_sequence(X, batch_sizes, batch_first=True, enforce_sorted=False)
        packed_y, _ = self.lstm(packed_X)
        pred_y, _ = pad_packed_sequence(packed_y, batch_first=True)
        dialog_embed = pred_y[:, -1, :]
        return dialog_embed

In [11]:
class BasicLSTM(nn.Module):
    def __init__(self, embed_size, h_1, h_2):
        super(BasicLSTM, self).__init__()
        self.token_lstm = TokenLSTM(embed_size, h_1, 1)
        self.sequence_lstm = SequenceLSTM(h_1, h_2, 1)
        self.fc = nn.Linear(h_2, 3)

    def forward(self, X, seq_lens, batch_sizes):
        X_1 = self.token_lstm(X, seq_lens)
        X_2 = self.sequence_lstm(X_1, batch_sizes)
        logits = self.fc(X_2)
        pred_y = F.log_softmax(logits, dim=-1)
        return pred_y

In [12]:
hidden_dim = 128    # Hidden size for LSTM
device = 'cpu'

In [ ]:
def train(
    model: BasicLSTM, 
    train_data: LstmDataset, 
    loss_function: nn.CrossEntropyLoss, 
    optimizer: torch.optim.Adam,
    num_epochs=5
) -> None:
    model.train()

    for epoch in tqdm(range(num_epochs)):
        total_loss = 0.0

        # Loads the whole training dataset.
        # Index here can be any value. We have it here to call the
        # __getitem__ method of Dataset. Index value is not used.
        X, y, seq_lens, batch_sizes = train_data[0]
        X = X.to(device)
        y = y.to(device)
        seq_lens = seq_lens.tolist()
        batch_sizes = batch_sizes.tolist()

        y_pred = model.forward(X, seq_lens, batch_sizes).squeeze()

        loss = loss_function(y_pred, y)
        optimizer.zero_grad()
        loss.backward()

        # To avoid exploding gradients
        clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

        avg_loss = total_loss / len(train_data)
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss:.4f}")

In [14]:
model = BasicLSTM(input_dim, hidden_dim, hidden_dim).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [156]:
train(model, train_data, loss_function, optimizer, seq_lens, batch_sizes, num_epochs=2)

 50%|█████     | 1/2 [01:21<01:21, 81.02s/it]

Epoch 1/2, Loss: 0.0005


100%|██████████| 2/2 [02:45<00:00, 82.72s/it]

Epoch 2/2, Loss: 0.0005


In [467]:
torch.save(model, "lstm_basic.pt")

Testing

In [23]:
model = torch.load("lstm_basic_0.001_300.pt", map_location=torch.device('cpu'))

/var/folders/jr/mx0gc_pj68gcg70k9gg7zd7h0000gn/T/ipykernel_60936/3378007388.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("lstm_basic_0.001_300.pt",

In [28]:
from sklearn.metrics import f1_score

X, y, seq_lens, batch_sizes = test_data[0]
y_pred_logits = model.forward(X, seq_lens, batch_sizes).squeeze()   # logits
y_pred = torch.max(y_pred_logits, axis=1)

In [29]:
f1_score(y.detach().numpy(), y_pred.indices.detach().numpy(), average='macro')

0.1688076416337286